# 767. Reorganize String

## Topic Alignment
- **Role Relevance**: Character frequency balancing appears in data pipeline design where you need to distribute tasks to avoid resource contention.
- **Scenario**: Similar to round-robin scheduling in distributed systems, load balancing in ML training clusters, and preventing data skew in batch processing.

## Metadata Summary
- Source: [LeetCode - Reorganize String](https://leetcode.com/problems/reorganize-string/)
- Tags: `Greedy`, `Heap`, `Hash Table`, `String`, `Counting`, `Sorting`
- Difficulty: Medium
- Recommended Priority: High

## Problem Statement
Given a string `s`, rearrange the characters of `s` so that any two adjacent characters are not the same.

Return any possible rearrangement of `s` or return `""` if not possible.

**Constraints:**
- `1 <= s.length <= 500`
- `s` consists of lowercase English letters.

## Progressive Hints
- Hint 1: Count the frequency of each character. If any character appears more than `(n + 1) // 2` times, it's impossible to reorganize.
- Hint 2: Use a greedy approach: always place the most frequent remaining character next, but skip it if it's the same as the previous character.
- Hint 3: A max heap can help you efficiently retrieve the most frequent character at each step.
- Hint 4: To avoid placing the same character twice, you can temporarily hold the previously placed character and add it back to the heap after placing the next different character.

## Solution Overview
The optimal approach uses a greedy strategy with a max heap:
1. Count the frequency of each character.
2. Check if reorganization is possible: the most frequent character should not exceed `(n + 1) // 2`.
3. Use a max heap to always get the most frequent character.
4. Build the result by alternating characters: pick the most frequent, then pick the next most frequent (different from the previous).
5. Use a temporary variable to hold the previously used character to prevent consecutive duplicates.

**Why greedy works**: By always placing the most frequent character first (when possible), we minimize the risk of being unable to place all instances of that character.

## Detailed Explanation
1. **Count frequencies**: Use a Counter or hash map to count each character's frequency.
2. **Early termination check**: If `max_freq > (n + 1) // 2`, return empty string (impossible).
3. **Build max heap**: Python's heapq is a min heap, so store negative counts to simulate max heap.
4. **Greedy placement**:
   - Pop the most frequent character from heap
   - Append it to result
   - Store it temporarily (don't add back to heap yet)
   - Pop the next most frequent character
   - Append it to result
   - Add the previous character back to heap if it still has remaining count
5. **Handle last character**: After the loop, there might be one character left in the temporary variable.

**Algorithm flow**:
```
heap = [(-count, char) for char, count in frequency.items()]
result = []
prev_count, prev_char = 0, ''

while heap:
    count, char = heappop(heap)
    result.append(char)
    
    if prev_count < 0:
        heappush(heap, (prev_count, prev_char))
    
    prev_count = count + 1
    prev_char = char
```

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Greedy with max heap | O(n log k) | O(k) | k is number of unique chars (max 26). Optimal solution. |
| Greedy with sorting | O(n log k) | O(k) | Sort by frequency, fill even positions first, then odd. |
| Backtracking | O(n!) | O(n) | Try all permutations. Too slow. |

## Reference Implementation

In [ ]:
import heapq
from collections import Counter


def reorganize_string(s: str) -> str:
    """
    Rearrange string so no two adjacent characters are the same.
    
    Args:
        s: Input string of lowercase letters
    
    Returns:
        Reorganized string or empty string if impossible
    """
    # Count character frequencies
    freq = Counter(s)
    n = len(s)
    
    # Check if reorganization is possible
    max_freq = max(freq.values())
    if max_freq > (n + 1) // 2:
        return ""
    
    # Build max heap (use negative counts for max heap)
    heap = [(-count, char) for char, count in freq.items()]
    heapq.heapify(heap)
    
    result = []
    prev_count, prev_char = 0, ''
    
    while heap:
        # Get most frequent character
        count, char = heapq.heappop(heap)
        result.append(char)
        
        # Add previous character back if it still has count
        if prev_count < 0:
            heapq.heappush(heap, (prev_count, prev_char))
        
        # Update previous character (decrement count)
        prev_count = count + 1  # +1 because count is negative
        prev_char = char
    
    return ''.join(result)

## Validation

In [ ]:
def is_valid_reorganization(s: str, result: str) -> bool:
    """Check if result is a valid reorganization of s."""
    if not result:
        return False
    if sorted(s) != sorted(result):
        return False
    for i in range(len(result) - 1):
        if result[i] == result[i + 1]:
            return False
    return True


cases = [
    ("aab", True),
    ("aaab", False),
    ("vvvlo", True),
    ("aabbcc", True),
    ("aaaa", False),
    ("a", True),
    ("ab", True),
]

for s, should_succeed in cases:
    result = reorganize_string(s)
    if should_succeed:
        assert is_valid_reorganization(s, result), f"Failed for s={s}: got {result}"
    else:
        assert result == "", f"Should be impossible for s={s}, but got {result}"

print('All tests passed for LC 767.')

## Complexity Analysis
- **Time Complexity**: O(n log k) where n is the length of string and k is the number of unique characters (max 26)
  - Counting frequencies: O(n)
  - Building heap: O(k)
  - Each character is pushed/popped from heap once: O(n log k)
  - Overall: O(n log k), but since k ≤ 26, effectively O(n)
- **Space Complexity**: O(k) for the heap and frequency counter (k ≤ 26, so O(1) in practice)
- **Bottleneck**: The heap operations dominate, but with k ≤ 26, it's very efficient.

## Edge Cases & Pitfalls
- **Impossible cases**: When the most frequent character appears more than `(n + 1) // 2` times, it's impossible to reorganize.
- **Single character**: Returns that character.
- **Two characters**: If equal frequency, alternates them. If one has higher frequency, check feasibility.
- **Off-by-one in feasibility check**: Use `(n + 1) // 2` not `n // 2` (for odd-length strings).
- **Heap sign**: Remember to use negative counts for max heap in Python.
- **Last character handling**: The last character in `prev_char` might not be added if heap is empty.

## Follow-up Variants
- What if we need to ensure no two adjacent characters are within k distance in the alphabet?
- How would you extend this to ensure no character appears twice within any window of size k?
- Can you solve this problem with a sorting-based approach instead of a heap?
- How would you handle the case where characters have different priorities beyond frequency?

## Takeaways
- Greedy algorithms with heaps are powerful for scheduling and arrangement problems.
- Always check feasibility first to avoid unnecessary computation.
- When you need to avoid adjacent duplicates, use a "cooldown" pattern: hold the previous item temporarily.
- The max heap pattern (using negative values in Python's min heap) is a common technique.
- For character frequency problems, remember that there are at most 26 unique lowercase letters, making many operations effectively O(1).

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 358 | Rearrange String k Distance Apart | Greedy + Heap + Cooldown |
| 621 | Task Scheduler | Greedy + Heap |
| 1054 | Distant Barcodes | Greedy + Heap |
| 984 | String Without AAA or BBB | Greedy |